# DataMate — phiên train t2 trên Colab

1. Upload cả thư mục `colab_t2` vào **My Drive** (đường dẫn `MyDrive/colab_t2`). Thư mục đã có `out/checkpoint-10` để chạy tiếp.
2. Runtime → Change runtime type → **T4 GPU**.
3. Chạy lần lượt các ô (ô 2 sẽ hỏi quyền Drive: bấm cho phép). Ô cuối tự chạy tiếp từ checkpoint mới nhất trong `MyDrive/colab_t2/out/`.
4. **Giữ tab mở**. Nếu bị ngắt: mở lại, chạy lại từ ô 1.
5. Kết quả cuối: `MyDrive/colab_t2/out/final_lora/`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.exists('/content/drive/MyDrive/colab_t2/sft_t2.jsonl'), 'upload thư mục colab_t2 vào My Drive trước'

In [ ]:
!pip install -q unsloth

In [ ]:
import os, glob, subprocess, sys
BASE = "/content/drive/MyDrive/colab_t2"
OUT = f"{BASE}/out"
ckpts = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
env = dict(os.environ,
           SFT_DATA_PATH=f"{BASE}/sft_t2.jsonl", INIT_ADAPTER=f"{BASE}/t1_lora", OUTPUT_DIR=OUT,
           EPOCHS="1", LR="7e-5", LOG_STEPS="5", SAVE_STEPS="10", EVAL_STEPS="0", MAX_SEQ_LEN="12288",
           TIME_LIMIT_H="11", PYTHONUNBUFFERED="1", RESUME_CKPT=ckpts[-1] if ckpts else "")
print("resume from:", env["RESUME_CKPT"] or "(fresh start from the t1 adapter)", flush=True)
p = subprocess.Popen([sys.executable, f"{BASE}/train_kaggle.py"], env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
print("exit code", p.wait())
print("final adapter:", os.listdir(f"{OUT}/final_lora") if os.path.isdir(f"{OUT}/final_lora") else "not yet")